# TPU v5e (Colab) + EC2 L40S: baselines and GapShift

The router runs in Colab next to vLLM-TPU. An outgoing SSH tunnel connects local
port 8001 directly to the EC2 GPU server. Start EC2 first with
`bash scripts/ec2/setup.sh` and `bash scripts/ec2/serve.sh` inside tmux.
Both engines must serve the same model, dtype and maximum context length.
Select a **TPU v5e** runtime. These cells launch real model servers and experiments.

This notebook requires the branch containing the EC2/GapShift changes. If they
are still local, upload that checkout to `/content/hetero-llm-inference` and set
`UPDATE_REPO=False`, or publish the branch before cloning it here.

In [ ]:
# 1. Repository and TPU environment
import os, subprocess, sys, time, json
from pathlib import Path
REPO = '/content/hetero-llm-inference'
BRANCH = 'keyi/ec2-gapshift'  # use the branch containing these changes
UPDATE_REPO = True

def run(args, **kwargs):
    return subprocess.run(args, check=True, **kwargs)

if not Path(REPO).exists():
    run(['git', 'clone', 'https://github.com/clemrho/hetero-llm-inference', REPO])
if UPDATE_REPO:
    run(['git', '-C', REPO, 'fetch', 'origin'])
    run(['git', '-C', REPO, 'checkout', BRANCH])
    run(['git', '-C', REPO, 'pull', '--ff-only'])
os.chdir(REPO)
assert Path('scripts/ec2/tunnel.sh').exists(), 'This checkout does not contain the EC2 changes'
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv', 'httpx', 'aiohttp'])
VENV = '/content/vllm_tpu_env'
if not Path(VENV + '/bin/python').exists():
    run(['uv', 'venv', '--python', '3.12', VENV])
if subprocess.run([VENV + '/bin/python', '-c', 'import vllm'], capture_output=True).returncode:
    run(['uv', 'pip', 'install', '--python', VENV + '/bin/python', 'vllm-tpu'])
run([VENV + '/bin/python', '-c', 'import jax, vllm; print(vllm.__version__); print(jax.devices())'])

In [ ]:
# 2. Model settings and TPU server (first XLA compilation can take 10+ minutes)
import urllib.request
MODEL = 'Qwen/Qwen3-4B'
MAX_LEN = 16384
TPU_URL, GPU_URL = 'http://127.0.0.1:8000', 'http://127.0.0.1:8001'
os.environ.update(MODEL=MODEL, TPU_URL=TPU_URL, GPU_URL=GPU_URL)
Path('logs').mkdir(exist_ok=True)
# Preserve process handles when re-running cells.
TPU_PROC = globals().get('TPU_PROC')
TUNNEL_PROC = globals().get('TUNNEL_PROC')

def served(port):
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{port}/v1/models', timeout=5) as response:
            return json.load(response)['data'][0]
    except Exception:
        return None

def start_tpu():
    global TPU_PROC
    current = served(8000)
    if current and current['id'] == MODEL and current.get('max_model_len') == MAX_LEN:
        print('TPU already ready:', current)
        return
    if current:
        if TPU_PROC is None or TPU_PROC.poll() is not None:
            raise RuntimeError('A different TPU server is running; stop that process before restarting')
        TPU_PROC.terminate()
        TPU_PROC.wait(timeout=60)
        time.sleep(10)
    log_path = Path(f'logs/vllm_tpu_{MAX_LEN}.log')
    with log_path.open('w') as logfile:
        TPU_PROC = subprocess.Popen([
            VENV + '/bin/vllm', 'serve', MODEL, '--host', '127.0.0.1', '--port', '8000',
            '--dtype', 'bfloat16', '--max-model-len', str(MAX_LEN),
            '--enable-prefix-caching', '--enable-prompt-tokens-details'],
            stdout=logfile, stderr=subprocess.STDOUT)
    start = time.monotonic()
    while served(8000) is None:
        if TPU_PROC.poll() is not None:
            raise RuntimeError(log_path.read_text()[-6000:])
        if time.monotonic() - start > 1800:
            raise TimeoutError(f'TPU startup timed out; inspect {log_path}')
        print(f'Waiting for TPU: {time.monotonic()-start:.0f}s', end='\r')
        time.sleep(10)
    print('\nTPU ready:', served(8000))

start_tpu()

In [ ]:
# 3. Direct EC2 SSH connection. The private key stays in this temporary runtime.
EC2_HOST = 'your-instance-public-dns'
EC2_USER = 'ubuntu'
EC2_PORT = 8000
EC2_KEY = '/content/.ssh/ec2.pem'

if not Path(EC2_KEY).exists():
    from google.colab import files
    uploaded = files.upload()  # choose the EC2 .pem file
    if len(uploaded) != 1:
        raise ValueError('Choose exactly one SSH private key')
    Path(EC2_KEY).parent.mkdir(parents=True, exist_ok=True)
    filename, content = next(iter(uploaded.items()))
    Path(EC2_KEY).write_bytes(content)
    original = Path('/content') / Path(filename).name
    if original.is_file() and original.resolve() != Path(EC2_KEY).resolve():
        original.unlink()
    del uploaded, content
Path(EC2_KEY).chmod(0o600)
if TUNNEL_PROC is not None and TUNNEL_PROC.poll() is None:
    TUNNEL_PROC.terminate()
    TUNNEL_PROC.wait(timeout=10)
if served(8001) is not None:
    raise RuntimeError('Local port 8001 is already occupied; stop the previous tunnel first')
env = {**os.environ, 'EC2_HOST': EC2_HOST, 'EC2_USER': EC2_USER,
       'EC2_PORT': str(EC2_PORT), 'EC2_KEY': EC2_KEY}
with open('logs/ec2_tunnel.log', 'w') as logfile:
    TUNNEL_PROC = subprocess.Popen(['bash', 'scripts/ec2/tunnel.sh'], env=env,
                                  stdout=logfile, stderr=subprocess.STDOUT)
for _ in range(30):
    if served(8001) is not None:
        break
    if TUNNEL_PROC.poll() is not None:
        raise RuntimeError(Path('logs/ec2_tunnel.log').read_text())
    time.sleep(2)
else:
    raise TimeoutError('EC2 tunnel did not become ready; inspect logs/ec2_tunnel.log')
print('EC2:', served(8001))

In [ ]:
# 4. Require identical model/length and working prefix caching on both engines
for port in (8000, 8001):
    info = served(port)
    if info is None or info['id'] != MODEL or info.get('max_model_len') != MAX_LEN:
        raise RuntimeError(f'Unexpected backend on :{port}: {info}; expected {MODEL}, {MAX_LEN}')
    run([sys.executable, 'scripts/check_backend.py', '--url', f'http://127.0.0.1:{port}', '--model', MODEL])

In [ ]:
# 5. Persistent-connection probes (do not subtract the historical PSC RTT)
import httpx, statistics
with httpx.Client(timeout=15) as client:
    for name, url in [('tpu', TPU_URL), ('ec2_l40s', GPU_URL)]:
        samples = []
        client.get(url + '/v1/models').raise_for_status()
        for _ in range(10):
            start = time.perf_counter()
            client.get(url + '/v1/models').raise_for_status()
            samples.append(time.perf_counter() - start)
        print(name, 'median probe ms:', 1000 * statistics.median(samples))

## Profiles and experiments

Profile only when both engines are idle, from this router location. The cold/warm
and decode samples include the current EC2 network path. Fresh prefixes isolate
samples and runs. Default routing uses history-based gap/size estimates; oracle
information is an explicit separate ablation. No new speedup is assumed.

In [ ]:
# 6. Collect the profiles needed by PROFILE and GAPSHIFT
PROFILE_PATH = f'{REPO}/logs/profiles_{MAX_LEN}.json'
lengths = sorted({1024, 2048, 4096, 8192, MAX_LEN - 256})
run([sys.executable, 'scripts/profile_backends.py', '--out', PROFILE_PATH,
     '--lengths', *map(str, lengths), '--reps', '3'])
# Optionally supply --gpu_cache_tokens / --tpu_cache_tokens from actual startup logs.

In [ ]:
# 7. Optional durable experiment copies
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/hetero_ec2_tpu'

In [ ]:
# 8. Generate/replay/analyze helpers; command failures stop the experiment
import glob, shutil

def run_experiment(name, gen_args, conc=(4, 8, 16), reps=3,
                   policies=('LB', 'FF', 'CA', 'RR', 'PROFILE', 'GAPSHIFT'), extra=()):
    for port in (8000, 8001):
        info = served(port)
        if not info or info['id'] != MODEL or info.get('max_model_len') != MAX_LEN:
            raise RuntimeError(f'Backend :{port} does not match the notebook model/length')
    folder = Path(REPO) / 'logs' / name
    folder.mkdir(parents=True, exist_ok=True)
    trace = folder / f'{name}.jsonl'
    try:
        run([sys.executable, 'traces/gen_synthetic.py', '--max_model_len', str(MAX_LEN),
             '--out', str(trace), *gen_args])
        run([sys.executable, 'replay/replay.py', '--trace', str(trace), '--sweep',
             '--policies', *policies, '--concurrency_list', *map(str, conc), '--reps', str(reps),
             '--profiles', PROFILE_PATH, '--log_dir', str(folder), *extra])
        logs = sorted(glob.glob(str(folder / name / '*_r*.jsonl')))
        with (folder / 'summary.json').open('w') as outfile:
            run([sys.executable, 'analysis/analyze.py', '--json', '--aggregate', *logs], stdout=outfile)
        run([sys.executable, 'analysis/analyze.py', *logs])
    finally:
        if SAVE_TO_DRIVE:
            shutil.copytree(folder, Path(DRIVE_DIR) / name, dirs_exist_ok=True)
            shutil.copy2(PROFILE_PATH, Path(DRIVE_DIR) / Path(PROFILE_PATH).name)
    return folder

SHORT = ['--steps', '3', '5', '--init_ctx', '2048', '8192',
         '--out_min', '20', '--out_max', '60', '--tool_min', '0.2', '--tool_max', '0.8']
LONG = ['--steps', '3', '5', '--init_ctx', '8192', '16384', '--obs_min', '1500',
        '--obs_max', '3000', '--out_min', '20', '--out_max', '60', '--tool_min', '0.5', '--tool_max', '2.0']

In [ ]:
# 9. Small pipeline check. This is not enough to claim a performance difference.
run_experiment('ec2_quick12', [*SHORT, '--per_combo', '3'], conc=(4, 8), reps=1)

In [ ]:
# 10. Short-context comparison, 64 workflows and three repetitions
run_experiment('ec2_short64', [*SHORT, '--per_combo', '16'], conc=(8, 16, 32), reps=3)

## Long-context and gap ablations

For long contexts, stop EC2's foreground server and restart it with
`MAX_LEN=32768 bash scripts/ec2/serve.sh`. Then run the following cell to restart
the TPU, re-run the backend checks and profiling cells, and run the long sweep.
The TPU may not support the chosen context at the desired concurrency; choose a
smaller common context limit and regenerate the trace if necessary.

In [ ]:
# 11. Run only after restarting EC2 with MAX_LEN=32768
MAX_LEN = 32768
start_tpu()
print('Re-run cells 4 and 6 to validate backends and rebuild profiles before the long sweep.')

In [ ]:
# 12. Long contexts (requires new profiles covering this range)
run_experiment('ec2_long16', [*LONG, '--per_combo', '4'], conc=(2, 4, 8), reps=3)

In [ ]:
# 13. Optional controlled ablations; each output directory records its configuration
# Choose a small concurrency first to estimate the runtime of the complete matrix.
for scale in (0, 0.25, 1, 4):
    run_experiment(f'ec2_gap_{MAX_LEN}_{scale}', [*SHORT, '--per_combo', '4'], conc=(8,),
                   policies=('CA', 'PROFILE', 'GAPSHIFT'), extra=('--gap_scale', str(scale)))
run_experiment(f'ec2_no_prewarm_{MAX_LEN}', [*SHORT, '--per_combo', '4'], conc=(8,),
               policies=('GAPSHIFT',), extra=('--max_prewarms', '0'))
run_experiment(f'ec2_oracle_{MAX_LEN}', [*SHORT, '--per_combo', '4'], conc=(8,),
               policies=('GAPSHIFT',), extra=('--gap_mode', 'oracle'))

In [ ]:
# 14. Close this notebook's tunnel when finished. The EC2 model server stays running.
if TUNNEL_PROC is not None and TUNNEL_PROC.poll() is None:
    TUNNEL_PROC.terminate()
    TUNNEL_PROC.wait(timeout=10)
print('Tunnel closed. Stop vLLM on EC2 and shut down the instance when your work is finished.')